In [2]:
import subprocess, time

# 1. Instalar dependência zstd (necessária para o instalador do Ollama)
!apt-get install -y zstd -q

# 2. Instalar o servidor Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Iniciar o servidor em background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Aguardar o servidor subir (importante!)
time.sleep(10)
print("🟢 Servidor Ollama iniciado!")

# 4. Instalar o SDK Python
!pip install ollama -q

# 5. Baixar o modelo base (só na primeira vez, ~800MB)
!ollama pull llama3.2:1b

# 6. Verificar
import ollama
print("✅ Tudo pronto! Servidor rodando e modelo baixado.")

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (456 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama gr

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import requests

def perguntar(modelo, prompt):
    url = "http://localhost:11434/api/generate"

    data = {
        "model": modelo,
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(url, json=data)
    return response.json()["response"]

In [4]:
import subprocess, time

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(10)

print("🟢 Servidor reiniciado!")

🟢 Servidor reiniciado!


Teste...

In [5]:
print(perguntar("llama3.2:1b", "Oi, tudo bem?"))

Oi, está tudo bem! Como posso ajudá-lo hoje? Você quer algo em específico ou apenas conversar um pouco?


Seguindo com o código

In [6]:
dataset = [

  {
    "pergunta": "Como faço um PIX?",
    "resposta": "Acesse a área de transferências, escolha PIX, informe a chave e confirme a operação."
  },
  {
    "pergunta": "Posso cancelar um PIX?",
    "resposta": "O PIX não pode ser cancelado após enviado. Se houver erro, é necessário pedir devolução ao destinatário."
  },
  {
    "pergunta": "O que é saldo disponível?",
    "resposta": "Saldo disponível é o valor que você pode usar imediatamente para pagamentos ou transferências."
  },
  {
    "pergunta": "Meu cartão foi bloqueado, o que faço?",
    "resposta": "Verifique no app o motivo do bloqueio e utilize a opção de desbloqueio ou solicite um novo cartão."
  },
  {
    "pergunta": "Perdi meu cartão",
    "resposta": "Bloqueie imediatamente pelo aplicativo para evitar uso indevido e solicite um novo cartão."
  },
  {
    "pergunta": "O que é CDI?",
    "resposta": "O CDI é uma taxa usada como referência para rendimentos de investimentos no mercado financeiro."
  },
  {
    "pergunta": "Como vejo minha fatura?",
    "resposta": "Acesse a área de cartões no aplicativo e selecione a opção de fatura para visualizar os detalhes."
  },
  {
    "pergunta": "Meu saldo sumiu",
    "resposta": "Verifique o extrato para identificar movimentações. Caso não reconheça, entre em contato com o suporte."
  },
  {
    "pergunta": "Como faço uma transferência?",
    "resposta": "Acesse transferências, informe os dados da conta de destino, o valor e confirme a operação."
  },
  {
    "pergunta": "É seguro usar o aplicativo do banco?",
    "resposta": "Sim, desde que você mantenha seus dados protegidos e nunca compartilhe senha ou códigos de verificação."
  }

]

In [7]:
SYSTEM_PROMPT = """
FROM llama3

SYSTEM "Você é o NuAssist, um assistente de banco digital. Responda sempre em português do Brasil, de forma clara, direta e segura. Nunca peça senha ou dados sensíveis."

PARAMETER temperature 0.3

MESSAGE user Como faço um PIX?
MESSAGE assistant Acesse transferências, escolha PIX, informe a chave e confirme.

MESSAGE user Posso cancelar um PIX?
MESSAGE assistant O PIX não pode ser cancelado após enviado. Solicite devolução ao destinatário.

MESSAGE user O que é saldo disponível?
MESSAGE assistant É o valor que você pode usar imediatamente.

MESSAGE user Meu cartão foi bloqueado
MESSAGE assistant Verifique o motivo no app e tente desbloquear.

MESSAGE user Perdi meu cartão
MESSAGE assistant Bloqueie imediatamente e solicite um novo cartão.

MESSAGE user O que é CDI?
MESSAGE assistant É uma taxa usada como referência para investimentos.

MESSAGE user Como vejo minha fatura?
MESSAGE assistant Acesse a área de cartões e visualize a fatura.

MESSAGE user Meu saldo sumiu
MESSAGE assistant Verifique o extrato e contate o suporte se necessário.

MESSAGE user Como faço transferência?
MESSAGE assistant Informe os dados da conta, valor e confirme.

MESSAGE user É seguro usar o app?
MESSAGE assistant Sim, desde que você não compartilhe seus dados.
"""

In [8]:
def gerar_modelfile(modelo_base, system_prompt, exemplos):
    linhas = []

    linhas.append(f"FROM {modelo_base}\n")
    linhas.append("PARAMETER temperature 0.6\n")

    linhas.append(f'SYSTEM """\n{system_prompt}\n"""\n')

    for ex in exemplos:
        linhas.append(f'MESSAGE user "{ex["pergunta"]}"')
        linhas.append(f'MESSAGE assistant "{ex["resposta"]}"\n')

    return "\n".join(linhas)

modelfile = gerar_modelfile("llama3.2:1b", SYSTEM_PROMPT, dataset)

with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)

In [9]:
!ollama create suporte-banco -f Modelfile

In [10]:
print(perguntar("suporte-banco", "Fizeram uma compra no meu cartão!"))

Acesse a área de transferências, informe os detalhes da transação (número do pedido, data, valor) e confirme.


In [17]:
import os
import ollama

def limpar_tela():
    os.system('cls' if os.name == 'nt' else 'clear')

def mostrar_menu():
    print("="*40)
    print(" 💳  NuAssist - Suporte Bancário")
    print("="*40)
    print("1 - Iniciar atendimento")
    print("2 - Limpar conversa")
    print("3 - Sobre")
    print("0 - Sair")
    print("="*40)

def sobre():
    print("\n📌 Sobre o sistema:")
    print("Este é um chatbot de suporte bancário usando IA (Ollama).")
    print("Responde dúvidas comuns de forma rápida e segura.\n")
    input("Pressione ENTER para voltar...")

def chatbot():
    historico = [{"role": "system", "content": SYSTEM_PROMPT}]

    while True:
        limpar_tela()
        mostrar_menu()
        opcao = input("Escolha uma opção: ")

        if opcao == "1":
            limpar_tela()
            print("💬 Atendimento iniciado (digite 'sair' para voltar)\n")

            while True:
                user = input("Você: ")

                if user.lower() == "sair":
                    break

                historico.append({"role": "user", "content": user})

                resposta = ollama.chat(
                    model="suporte-banco",
                    messages=historico
                )

                texto = resposta["message"]["content"]
                print("🤖 Bot:", texto)

                historico.append({"role": "assistant", "content": texto})

            input("\nPressione ENTER para voltar ao menu...")

        elif opcao == "2":
            historico = [{"role": "system", "content": SYSTEM_PROMPT}]
            print("\n🧹 Conversa resetada!")
            input("Pressione ENTER...")

        elif opcao == "3":
            limpar_tela()
            sobre()

        elif opcao == "0":
            print("\nEncerrando sistema...")
            break

        else:
            print("\n❌ Opção inválida!")
            input("Pressione ENTER...")

# rodar
chatbot()

 💳  NuAssist - Suporte Bancário
1 - Iniciar atendimento
2 - Limpar conversa
3 - Sobre
0 - Sair
Escolha uma opção: 1
💬 Atendimento iniciado (digite 'sair' para voltar)

Você: sair

Pressione ENTER para voltar ao menu...sair
 💳  NuAssist - Suporte Bancário
1 - Iniciar atendimento
2 - Limpar conversa
3 - Sobre
0 - Sair
Escolha uma opção: 0

Encerrando sistema...
